In [1]:
from models.WatermarkNet import WatermarkNet
from pathlib import Path
from run_evaluate import *
import exp_setup as exp_setup
import torchaudio
from exp.ExpEmbedWatermark import ExpEmbedWatermark, ExpConfig as ExpEmbedWatermarkCfg

In [2]:
exp_cfg = exp_setup.init_config()
# speaker_audio_wm_dic = get_speaker_audio_wm_dic(exp_cfg)

In [3]:
exp_cfg.wm_length

16

In [4]:
speakers_wm_lst, benign_org_wm, benign_encoded_wm = exp_setup.get_speakers_and_wm(exp_cfg, exp_cfg.wm_length)
wm_pool, speaker_idx = get_wm_pool('VCTK_p261', speakers_wm_lst)
wm_pool[speaker_idx]


original watermarks maximum overlapping = 0.75; wm_len = 16
Do not use hamming code for watermarks.


tensor([1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1], device='cuda:0')

In [5]:
cfg = ExpEmbedWatermarkCfg(
        extra_audio_lst_dic=exp_setup.process_Obama_voice(exp_cfg),     # add Obama's speech

        speakers_wm_lst=speakers_wm_lst,
        vctk_dir=exp_setup.get_vctk_dir(exp_cfg),

        benign_org_wm=np.array(wm_pool[speaker_idx].cpu()),
        benign_encoded_wm=np.array(wm_pool[speaker_idx].cpu()),

        wav2vec2_dir=exp_cfg.wav2vec2_pretrained_dir,

        aug_normal_prob=exp_cfg.wmnet_aug_normal_prob,
        aug_normal_scale=exp_cfg.wmnet_aug_normal_scale,
    )

aug_normal_prob = 0.40, aug_normal_scale = 0.04.


In [6]:
from models.ModelTrainer import ModelTrainer
audio_sec_len = 16000           # by default, WatermarkNet split audio into 1-second sections
device = 'cuda'
wm_net = WatermarkNet(cfg.benign_encoded_wm, cfg.expected_sr, cfg.audio_sec_len,
                        wav2vec2_dir=cfg.wav2vec2_dir,
                        aug_normal_prob=cfg.aug_normal_prob, aug_normal_scale=cfg.aug_normal_scale)
model_dir = Path(exp_cfg.out_dir).joinpath("ExpEmbedWatermark")
dic_saved, return_path = ModelTrainer.load_latest_ckpt(model_dir.joinpath("ckpt"),return_path=True)
wm_net.load_state_dict(dic_saved["model_state"])
wm_net = wm_net.to(device)
wm_net.eval()



loading check point from file: /home/binhaoma/code/AudioMarkNet_artifacts/out/ExpEmbedWatermark/ckpt/saved_epoch-169.tar, epoch_check = 169


/home/binhaoma/code/AudioMarkNet_artifacts/amn_code/models/ModelTrainer.py:231: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  dic_saved = torch.load(ckpt_path)


WatermarkNet(
  (encoder): MyWatermarkEncoder(
    (wav2vec2): Wav2Vec2Model(
      (feature_extractor): FeatureExtractor(
        (conv_layers): ModuleList(
          (0): ConvLayerBlock(
            (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
            (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
          )
          (1-4): 4 x ConvLayerBlock(
            (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
          )
          (5-6): 2 x ConvLayerBlock(
            (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
          )
        )
      )
      (encoder): Encoder(
        (feature_projection): FeatureProjection(
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (projection): Linear(in_features=512, out_features=1024, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (transformer): Transformer(
          (pos_conv_embed): ConvolutionalPosit

In [8]:
# root_dir = Path("sub_vctk_clean_testset_wav")
# root_dir

In [9]:

# waveform, sr = torchaudio.load("../sub_vctk_clean_testset_wav/p261/p261_001_mic1.wav")  # waveform.shape = [channels, time]

# tgt_wm = wm_pool[speaker_idx]

# # 转为 numpy：先移到 CPU（以防是 GPU tensor），再 .numpy()
# benign_wav = waveform.cpu().numpy().squeeze(0)

# wav_secs = wm_net.split_waveform(benign_wav) # benign wav 切片
        
# tgt_wm = tgt_wm.unsqueeze(0).to(device) #message 转numpy
# tgt_wm = tgt_wm.repeat(wav_secs.shape[0], 1)


# logits, mask = wm_net((wav_secs, tgt_wm), return_mask=True)
# preds = (logits > 0.0).long()
# acc_wm = (preds == tgt_wm).sum() / preds.numel()
# acc_wm = acc_wm.item()

# acc_wm


In [ ]:
# 生成wm 音频---------------------------------------
import torchaudio
import os
# 指定根目录
root_dir = "./sub_vctk_clean_testset_wav"
save_dir = 'wm_sub_vctk_clean_testset_wav'

# 遍历所有子目录和文件
speaker_dir = os.listdir(root_dir)

for speaker in speaker_dir:
    speaker_specific_dir = os.path.join(root_dir, speaker)
    audios = os.listdir(speaker_specific_dir)
    save_specific_dir = os.path.join(save_dir, speaker)
    wm_pool, speaker_idx = get_wm_pool(f'VCTK_{speaker}', speakers_wm_lst)
    for audio in audios:
        audio_path = os.path.join(speaker_specific_dir, audio)
        wav,sr = torchaudio.load(audio_path)
        x = wav.cpu().numpy().squeeze(0)
        wav_secs = wm_net.split_waveform(x)
        acc_wm, acc_org, wm_waveform = wm_net.test_wm_capability(x, np.array(wm_pool[speaker_idx].cpu()))
        save_wm_wav = wm_waveform.flatten().cpu().detach().numpy()[:len(wav[0])]
        if acc_wm<1:
            print('----------------------------------------')
            print(sr)
            print('audio_name is', audio)
            print(speaker)
            print(wm_pool[speaker_idx])
            print('acc is------------', acc_wm)
            print('----------------------------------------')

        # 确保父目录存在
        output_path = Path(save_specific_dir) / audio
        output_path.parent.mkdir(parents=True, exist_ok=True)
        torchaudio.save(str(output_path), torch.from_numpy(save_wm_wav).unsqueeze(0).cpu(),sr)
        

----------------------------------------
16000
audio_name is p248_014_mic1.wav
p248
tensor([0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0], device='cuda:0')
acc is------------ 0.9553571939468384
----------------------------------------
----------------------------------------
16000
audio_name is p302_175_mic1.wav
p302
tensor([0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1], device='cuda:0')
acc is------------ 0.875
----------------------------------------
----------------------------------------
16000
audio_name is p302_010_mic1.wav
p302
tensor([0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1], device='cuda:0')
acc is------------ 0.9375
----------------------------------------
----------------------------------------
16000
audio_name is p302_129_mic1.wav
p302
tensor([0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1], device='cuda:0')
acc is------------ 0.953125
----------------------------------------
----------------------------------------
16000
audio_name is p302_124_mic1.wav
p302
ten

In [33]:
# pitch shift --------------------------------------

import python_stretch as ps
import torchaudio
import os
import torch

stretch = ps.Signalsmith.Stretch()
tar_sr = 16000
stretch.preset(1, tar_sr)  # 设置通道数和采样率


semitones = -0.8
def custom_map(f):
    return f * (2 ** (semitones / 12))  # 上升 5 个 semitones

stretch.setFreqMap(custom_map)


# 指定根目录
root_dir = "./pitch_shift_wav"
save_dir = 'recover_audio'

# 遍历所有子目录和文件
speaker_dir = os.listdir(root_dir)

for speaker in speaker_dir:
    speaker_specific_dir = os.path.join(root_dir, speaker)
    audios = os.listdir(speaker_specific_dir)
    save_specific_dir = os.path.join(save_dir, speaker)
    for audio in audios:
        audio_path = os.path.join(speaker_specific_dir, audio)
        wav,sr = torchaudio.load(audio_path)
        wav_process = stretch.process(wav)
        # 确保父目录存在
        output_path = Path(save_specific_dir) / audio
        output_path.parent.mkdir(parents=True, exist_ok=True)
        torchaudio.save(str(output_path), torch.from_numpy(wav_process),sr)
        

In [ ]:
#--------------------------------------- 验证wm ---------------------------------------

# 生成wm 音频---------------------------------------
import torchaudio
import os
# 指定根目录
root_dir = "./pitch_shift_wav"

# 遍历所有子目录和文件
speaker_dir = os.listdir(root_dir)
acc_list = []
for speaker in speaker_dir:
    speaker_specific_dir = os.path.join(root_dir, speaker)
    audios = os.listdir(speaker_specific_dir)
    save_specific_dir = os.path.join(save_dir, speaker)
    wm_pool, speaker_idx = get_wm_pool(f'VCTK_{speaker}', speakers_wm_lst)
    for audio in audios:
        audio_path = os.path.join(speaker_specific_dir, audio)
        wav,sr = torchaudio.load(audio_path)

        x = wav.cpu().numpy().squeeze(0)
        wav_secs = wm_net.split_waveform(x)

        logits = wm_net.decoder(wm_net.spec_for_classificiation(wav_secs)[..., wm_net.min_freq_idx: wm_net.max_freq_idx, :])
        pred_wm = (logits > 0.0).long()

        acc_wm = (pred_wm == wm_pool[speaker_idx]).sum() / pred_wm.numel()

        print('----------------------------------')



        print('----------------------------------')

        acc_list.append(acc_wm)
        print(acc_wm)
        


        

tensor(0.4219, device='cuda:0')
tensor(0.4375, device='cuda:0')
tensor(0.3958, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.4167, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.4167, device='cuda:0')
tensor(0.4062, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0

In [31]:
sum(acc_list)/len(acc_list)

tensor(0.4500, device='cuda:0')

In [29]:
#--------------------------------------- 验证wm ---------------------------------------

# 生成wm 音频---------------------------------------
import torchaudio
import os
# 指定根目录
root_dir = "./sub_vctk_clean_testset_wav"

# 遍历所有子目录和文件
speaker_dir = os.listdir(root_dir)
acc_list_clean = []
for speaker in speaker_dir:
    speaker_specific_dir = os.path.join(root_dir, speaker)
    audios = os.listdir(speaker_specific_dir)
    save_specific_dir = os.path.join(save_dir, speaker)
    wm_pool, speaker_idx = get_wm_pool(f'VCTK_{speaker}', speakers_wm_lst)
    for audio in audios:
        audio_path = os.path.join(speaker_specific_dir, audio)
        wav,sr = torchaudio.load(audio_path)

        x = wav.cpu().numpy().squeeze(0)
        wav_secs = wm_net.split_waveform(x)

        logits = wm_net.decoder(wm_net.spec_for_classificiation(wav_secs)[..., wm_net.min_freq_idx: wm_net.max_freq_idx, :])
        pred_wm = (logits > 0.0).long()

        acc_wm = (pred_wm == wm_pool[speaker_idx]).sum() / pred_wm.numel()

        acc_list_clean.append(acc_wm)
        print(acc_wm)
        


        

tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0.3750, device='cuda:0')
tensor(0

In [30]:
sum(acc_list_clean)/len(acc_list_clean)

tensor(0.4151, device='cuda:0')

In [34]:
#--------------------------------------- 验证wm ---------------------------------------

# 生成wm 音频---------------------------------------
import torchaudio
import os
# 指定根目录
root_dir = "./recover_audio"

# 遍历所有子目录和文件
speaker_dir = os.listdir(root_dir)
acc_recover_list = []
for speaker in speaker_dir:
    speaker_specific_dir = os.path.join(root_dir, speaker)
    audios = os.listdir(speaker_specific_dir)
    save_specific_dir = os.path.join(save_dir, speaker)
    wm_pool, speaker_idx = get_wm_pool(f'VCTK_{speaker}', speakers_wm_lst)
    for audio in audios:
        audio_path = os.path.join(speaker_specific_dir, audio)
        wav,sr = torchaudio.load(audio_path)

        x = wav.cpu().numpy().squeeze(0)
        wav_secs = wm_net.split_waveform(x)

        logits = wm_net.decoder(wm_net.spec_for_classificiation(wav_secs)[..., wm_net.min_freq_idx: wm_net.max_freq_idx, :])
        pred_wm = (logits > 0.0).long()

        acc_wm = (pred_wm == wm_pool[speaker_idx]).sum() / pred_wm.numel()

        print('----------------------------------')



        print('----------------------------------')

        acc_recover_list.append(acc_wm)
        print(acc_wm)
        


        

----------------------------------
----------------------------------
tensor(0.9219, device='cuda:0')
----------------------------------
----------------------------------
tensor(1., device='cuda:0')
----------------------------------
----------------------------------
tensor(0.9375, device='cuda:0')
----------------------------------
----------------------------------
tensor(0.8958, device='cuda:0')
----------------------------------
----------------------------------
tensor(1., device='cuda:0')
----------------------------------
----------------------------------
tensor(0.8958, device='cuda:0')
----------------------------------
----------------------------------
tensor(0.9375, device='cuda:0')
----------------------------------
----------------------------------
tensor(0.7917, device='cuda:0')
----------------------------------
----------------------------------
tensor(0.9000, device='cuda:0')
----------------------------------
----------------------------------
tensor(1., device='c

In [35]:
sum(acc_recover_list)/len(acc_recover_list)

tensor(0.9599, device='cuda:0')

In [36]:
#--------------------------------------- 验证wm ---------------------------------------

# 生成wm 音频---------------------------------------
import torchaudio
import os
# 指定根目录
root_dir = "./wm_sub_vctk_clean_testset_wav"

# 遍历所有子目录和文件
speaker_dir = os.listdir(root_dir)
acc_wm_list = []
for speaker in speaker_dir:
    speaker_specific_dir = os.path.join(root_dir, speaker)
    audios = os.listdir(speaker_specific_dir)
    save_specific_dir = os.path.join(save_dir, speaker)
    wm_pool, speaker_idx = get_wm_pool(f'VCTK_{speaker}', speakers_wm_lst)
    for audio in audios:
        audio_path = os.path.join(speaker_specific_dir, audio)
        wav,sr = torchaudio.load(audio_path)

        x = wav.cpu().numpy().squeeze(0)
        wav_secs = wm_net.split_waveform(x)

        logits = wm_net.decoder(wm_net.spec_for_classificiation(wav_secs)[..., wm_net.min_freq_idx: wm_net.max_freq_idx, :])
        pred_wm = (logits > 0.0).long()

        acc_wm = (pred_wm == wm_pool[speaker_idx]).sum() / pred_wm.numel()

        print('----------------------------------')



        print('----------------------------------')

        acc_wm_list.append(acc_wm)
        print(acc_wm)
        


        

----------------------------------
----------------------------------
tensor(0.9219, device='cuda:0')
----------------------------------
----------------------------------
tensor(1., device='cuda:0')
----------------------------------
----------------------------------
tensor(0.9375, device='cuda:0')
----------------------------------
----------------------------------
tensor(0.9167, device='cuda:0')
----------------------------------
----------------------------------
tensor(1., device='cuda:0')
----------------------------------
----------------------------------
tensor(0.8958, device='cuda:0')
----------------------------------
----------------------------------
tensor(0.9375, device='cuda:0')
----------------------------------
----------------------------------
tensor(0.7917, device='cuda:0')
----------------------------------
----------------------------------
tensor(0.9125, device='cuda:0')
----------------------------------
----------------------------------
tensor(1., device='c

In [37]:
sum(acc_wm_list)/len(acc_wm_list)

tensor(0.9625, device='cuda:0')

In [ ]:
dirnames

[]

In [ ]:
import torchaudio

# 加载音频：返回 waveform 和 sample rate
waveform, sr = torchaudio.load("../out/ExpEmbedWatermark/VCTK_p261/p261_001_mic1.wav")  # waveform.shape = [channels, time]

# 转为 numpy：先移到 CPU（以防是 GPU tensor），再 .numpy()
x = waveform.cpu().numpy().squeeze(0)

print(x.shape)  # (1, time) 表示单声道；(2, time) 表示立体声

wav_secs = wm_net.split_waveform(x)

logits = wm_net.decoder(wm_net.spec_for_classificiation(wav_secs)[..., wm_net.min_freq_idx: wm_net.max_freq_idx, :])
pred_wm = (logits > 0.0).long()
pred_wm

(36950,)


tensor([[1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1],
        [1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1],
        [0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1]], device='cuda:0')

In [ ]:
acc_wm = (pred_wm == wm_pool[speaker_idx]).sum() / pred_wm.numel()
acc_wm = acc_wm.item()
acc_wm

0.8333333730697632

In [ ]:
# org_preds = wm_net.inference(x, wm_pool, benign_wm_included=False)
# org_preds

In [ ]:
acc_wm, acc_org, wm_waveform = wm_net.test_wm_capability(x, np.array([1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1]))

In [ ]:
org_preds = wm_net.inference(wm_waveform, wm_pool, benign_wm_included=True)
org_preds

In [ ]:
wm_waveform.shape

torch.Size([3, 16000])

In [ ]:

logits = wm_net.decoder(wm_net.spec_for_classificiation(wm_waveform)[..., wm_net.min_freq_idx: wm_net.max_freq_idx, :])
pred_wm = (logits > 0.0).long()
pred_wm

tensor([[1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1],
        [1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1],
        [1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1]], device='cuda:0')

In [ ]:
wm_waveform.flatten().cpu().detach().numpy().shape

(48000,)

In [ ]:
wm_pool, speaker_idx = get_wm_pool('VCTK_p261', speakers_wm_lst)
wm_pool[speaker_idx]

tensor([1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1], device='cuda:0')

In [ ]:
speakers_wm_lst

[{'speaker': 'VCTK_p261',
  'org_wm': array([1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1]),
  'encoded_wm': array([1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1])},
 {'speaker': 'VCTK_p225',
  'org_wm': array([1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0]),
  'encoded_wm': array([1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0])},
 {'speaker': 'VCTK_p294',
  'org_wm': array([0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0]),
  'encoded_wm': array([0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0])},
 {'speaker': 'VCTK_p347',
  'org_wm': array([1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0]),
  'encoded_wm': array([1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0])},
 {'speaker': 'VCTK_p238',
  'org_wm': array([1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1]),
  'encoded_wm': array([1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1])},
 {'speaker': 'VCTK_p234',
  'org_wm': array([0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0]),
  'encoded_wm': array([0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0

In [ ]:
# 加载音频：返回 waveform 和 sample rate
waveform, sr = torchaudio.load("../sub_vctk_clean_testset_wav/p261/p261_001_mic1.wav")  # waveform.shape = [channels, time]

# 转为 numpy：先移到 CPU（以防是 GPU tensor），再 .numpy()
x = waveform.cpu().numpy().squeeze(0)


import numpy as np
bits = np.array(np.random.randint(0, 2, size=exp_cfg.wm_length).tolist())
# bits = np.array(np.random.randint(0, 2, size=exp_cfg.wm_length))
# print(bits)
acc_wm, acc_org, wm_waveform = wm_net.test_wm_capability(x, np.array([1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1]))
acc_wm, acc_org

(1.0, 0.5)

In [ ]:
waveform_first = wm_waveform.flatten()[:len(waveform[0])]

acc_wm, acc_org, wm_waveform = wm_net.test_wm_capability(waveform_first.cpu(), np.array([1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1]))

acc_wm, acc_org, wm_waveform = wm_net.test_wm_capability(waveform_first.cpu(), np.array([1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1]))

acc_wm, acc_org, wm_waveform = wm_net.test_wm_capability(waveform_first.cpu(), np.array([1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1]))

acc_wm, acc_org

(0.953125, 0.5625)

In [ ]:

logits = wm_net.decoder(wm_net.spec_for_classificiation(wm_waveform)[..., wm_net.min_freq_idx: wm_net.max_freq_idx, :])
pred_wm = (logits > 0.0).long()
pred_wm

tensor([[1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0],
        [1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0],
        [0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0],
        [1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0]], device='cuda:0')

In [ ]:
wm_waveform.flatten()[:len(waveform[0])]


torch.Size([50921])

In [ ]:
wav,sr = torchaudio.load('out/ExpEmbedWatermark/VCTK_p261/p261_449_mic1.wav')
sr

16000

In [ ]:
exp_dir = Path(exp_cfg.out_dir).joinpath("ExpEvaluate")
exp_dir.mkdir(exist_ok=True)

our_saved_path = Path(exp_cfg.out_dir).joinpath(f"ExpEmbedWatermark/tgt_samples.bin")
# wavmark_saved_path = Path(exp_cfg.out_dir).joinpath(f"ExpWavMark/tgt_samples.bin")

################################################
# We do not evaluate wavmark anymore.
# A quick fix for existing code here is to make load the same data from our experiments
wavmark_saved_path = our_saved_path

################################################

with open(our_saved_path, 'rb') as handle:
    our_tgt_samples = pickle.load(handle)
with open(wavmark_saved_path, 'rb') as handle:
    wavmark_tgt_samples = pickle.load(handle)

# firstly, load all the audio
# speaker_audio_wm_dic = get_speaker_audio_wm_dic(exp_cfg)

In [ ]:
our_tgt_samples[0] # /home/wzong/My Passport/projects/data/vctk/wav48_silence_trimmed/p261/p261_069_mic1.flac

[{'text': 'There could be a way back.\n',
  'audio_file': '/home/wzong/My Passport/projects/data/vctk/wav48_silence_trimmed/p261/p261_069_mic1.flac',
  'speaker_name': 'VCTK_p261',
  'root_path': '/home/wzong/My Passport/projects/data/vctk',
  'language': 'en',
  'audio_unique_name': 'vctk#wav48_silence_trimmed/p261/p261_069_mic1',
  'audio_file_wm': PosixPath('/home/wzong/My Passport/projects/FakeAudioDetection/out/ExpEmbedWatermark/VCTK_p261/p261_069_mic1.wav')},
 array([1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1])]

In [ ]:
# speaker_audio_wm_dic[speaker_name].append({ # VCTK_p261
#             "org_audio": org_audio, # oriaudio
#             "our_wm_audio": our_wm_audio, "our_wm": our_wm, #[1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1]
#             "wavmark_wm_audio": wavmark_wm_audio, "wavmark_wm": wavmark_wm,
#         })